# 03 — Closed and Discriminative Sequential Patterns

This notebook takes the frequent sequential patterns mined separately from the positive and negative cohorts in Notebook 02.

The goals are:
1. Identify **closed sequential patterns** so redundant, less-informative patterns can be reduced.
2. Compare positive and negative support for the remaining patterns.
3. Extract **discriminative patterns** that occur substantially more often in pre-sepsis sequences than in non-sepsis sequences.
4. Save the resulting pattern sets for Notebook 04.

The notebook uses the frequent-pattern CSV files produced by Notebook 02. It does not rerun PrefixSpan.


In [ ]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs").exists() and (PROJECT_ROOT.parent / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PATTERN_DIR = PROJECT_ROOT / "outputs" / "patterns"
FIGURE_DIR = PATTERN_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

POSITIVE_FILE = PATTERN_DIR / "frequent_positive_patterns.csv"
NEGATIVE_FILE = PATTERN_DIR / "frequent_negative_patterns.csv"

# Main filtering controls
MIN_DISCRIMINATIVE_SUPPORT = 0.01
MIN_SUPPORT_DIFFERENCE = 0.05
MIN_SUPPORT_RATIO = 1.5

# Keep the final pattern set manageable for distance-feature generation.
MAX_DISCRIMINATIVE_PATTERNS = 300

print("Project root:", PROJECT_ROOT)
print("Pattern directory:", PATTERN_DIR)


In [ ]:
# Load frequent patterns from Notebook 02
if not POSITIVE_FILE.exists():
    raise FileNotFoundError(f"Missing: {POSITIVE_FILE}")

if not NEGATIVE_FILE.exists():
    raise FileNotFoundError(f"Missing: {NEGATIVE_FILE}")

positive_df = pd.read_csv(POSITIVE_FILE)
negative_df = pd.read_csv(NEGATIVE_FILE)

print("Positive frequent patterns:", len(positive_df))
print("Negative frequent patterns:", len(negative_df))

display(positive_df.head())
display(negative_df.head())


## 1. Normalize pattern representation

The pattern CSV may contain a Python-style representation in the `pattern` column. We convert it into a canonical tuple representation so that patterns from the two cohorts can be compared reliably.

For example, a pattern can conceptually look like:

`('HR_HIGH', 'MAP_LOW')`

or, when an hourly event contains multiple symbols, the representation may contain tuples/itemsets.

The code below deliberately preserves the mined representation rather than inventing new clinical groupings.


In [ ]:
def parse_pattern(value):
    if isinstance(value, (tuple, list)):
        return tuple(value)

    if pd.isna(value):
        return tuple()

    text = str(value).strip()
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (tuple, list)):
            return tuple(parsed)
        return (parsed,)
    except (ValueError, SyntaxError):
        # Fallback for simple string representations
        return (text,)

def canonical_pattern(value):
    return tuple(str(x) for x in parse_pattern(value))

positive_df["pattern_key"] = positive_df["pattern"].apply(canonical_pattern)
negative_df["pattern_key"] = negative_df["pattern"].apply(canonical_pattern)

positive_df["length"] = positive_df["pattern_key"].apply(len)
negative_df["length"] = negative_df["pattern_key"].apply(len)

print("Example normalized positive patterns:")
display(positive_df[["pattern", "pattern_key", "length", "support"]].head())


## 2. Combine positive and negative support

A pattern can be frequent in the positive cohort, the negative cohort, or both.

For discriminative analysis we need:

- positive support
- negative support
- support difference
- positive/negative support ratio

Patterns that have high positive support and substantially lower negative support are more useful for the proposed pre-sepsis pattern representation.


In [ ]:
pos_support = (
    positive_df[["pattern_key", "support"]]
    .drop_duplicates("pattern_key")
    .rename(columns={"support": "positive_support"})
)

neg_support = (
    negative_df[["pattern_key", "support"]]
    .drop_duplicates("pattern_key")
    .rename(columns={"support": "negative_support"})
)

support_df = pd.merge(
    pos_support,
    neg_support,
    on="pattern_key",
    how="outer"
)

support_df["positive_support"] = support_df["positive_support"].fillna(0.0)
support_df["negative_support"] = support_df["negative_support"].fillna(0.0)

support_df["support_difference"] = (
    support_df["positive_support"] - support_df["negative_support"]
)

# Ratio is defined with a small epsilon to avoid division by zero.
EPS = 1e-9
support_df["support_ratio"] = (
    support_df["positive_support"] /
    (support_df["negative_support"] + EPS)
)

support_df["length"] = support_df["pattern_key"].apply(len)

def pattern_to_text(pattern):
    return " -> ".join(pattern)

support_df["pattern_text"] = support_df["pattern_key"].apply(pattern_to_text)

display(
    support_df.sort_values(
        ["support_difference", "positive_support"],
        ascending=False
    ).head(20)
)


## 3. Closed-pattern filtering

A sequential pattern is considered **closed** when there is no strict super-pattern with the same support.

This notebook performs closed filtering separately within each cohort using the frequent patterns already mined by PrefixSpan.

Because Notebook 02 uses a bounded maximum pattern length, closedness is evaluated over the available mined pattern set. This is an implementation-level closed-pattern filter rather than a replacement for a dedicated CloSpan implementation.


In [ ]:
def is_subsequence(shorter, longer):
    """Return True if `shorter` occurs in `longer` in order."""
    if len(shorter) >= len(longer):
        return False

    i = 0
    for item in longer:
        if i < len(shorter) and item == shorter[i]:
            i += 1
    return i == len(shorter)

def closed_patterns(df):
    """
    Keep patterns for which there is no strict super-pattern
    with exactly the same support within the supplied mined set.
    """
    work = (
        df[["pattern_key", "support"]]
        .drop_duplicates("pattern_key")
        .copy()
    )

    patterns = list(work["pattern_key"])
    supports = dict(zip(work["pattern_key"], work["support"]))

    closed = []

    # Group candidates by support. A super-pattern can only invalidate
    # closedness if its support is identical.
    support_groups = {}
    for pattern, support in supports.items():
        support_groups.setdefault(round(float(support), 12), []).append(pattern)

    for pattern in patterns:
        support_key = round(float(supports[pattern]), 12)
        candidates = support_groups.get(support_key, [])

        has_same_support_superpattern = any(
            is_subsequence(pattern, candidate)
            for candidate in candidates
        )

        if not has_same_support_superpattern:
            closed.append(pattern)

    return work[work["pattern_key"].isin(closed)].copy()

positive_closed = closed_patterns(positive_df)
negative_closed = closed_patterns(negative_df)

print(f"Positive frequent patterns: {len(positive_df):,}")
print(f"Positive closed patterns:   {len(positive_closed):,}")
print(f"Negative frequent patterns: {len(negative_df):,}")
print(f"Negative closed patterns:   {len(negative_closed):,}")


## 4. Build the closed-pattern comparison set

For the next stage, we keep patterns that are closed in at least one cohort. Their positive and negative supports are then recomputed from the combined support table.

This makes the filtering explicit and reproducible instead of silently dropping patterns that only occur in one cohort.


In [ ]:
closed_keys = set(positive_closed["pattern_key"]) | set(negative_closed["pattern_key"])

closed_support_df = support_df[
    support_df["pattern_key"].isin(closed_keys)
].copy()

closed_support_df = closed_support_df.sort_values(
    ["support_difference", "positive_support"],
    ascending=False
).reset_index(drop=True)

print("Closed comparison patterns:", len(closed_support_df))
display(closed_support_df.head(20))


## 5. Discriminative pre-sepsis patterns

A simple, interpretable discriminative rule is used:

- minimum positive support ≥ `MIN_DISCRIMINATIVE_SUPPORT`
- positive-minus-negative support ≥ `MIN_SUPPORT_DIFFERENCE`
- positive/negative support ratio ≥ `MIN_SUPPORT_RATIO`

These are project parameters rather than universal clinical thresholds. They can be changed during the ablation stage.

The filtering is directional: the notebook is looking specifically for patterns associated more strongly with the positive pre-sepsis cohort.


In [ ]:
discriminative_df = closed_support_df[
    (closed_support_df["positive_support"] >= MIN_DISCRIMINATIVE_SUPPORT) &
    (closed_support_df["support_difference"] >= MIN_SUPPORT_DIFFERENCE) &
    (closed_support_df["support_ratio"] >= MIN_SUPPORT_RATIO)
].copy()

discriminative_df = discriminative_df.sort_values(
    ["support_difference", "positive_support", "support_ratio"],
    ascending=False
).reset_index(drop=True)

print("Discriminative patterns before cap:", len(discriminative_df))

if len(discriminative_df) > MAX_DISCRIMINATIVE_PATTERNS:
    discriminative_df = discriminative_df.head(MAX_DISCRIMINATIVE_PATTERNS).copy()
    print(f"Keeping top {MAX_DISCRIMINATIVE_PATTERNS} patterns for downstream distance features.")

print("Final discriminative patterns:", len(discriminative_df))

display(
    discriminative_df[
        [
            "pattern_text",
            "length",
            "positive_support",
            "negative_support",
            "support_difference",
            "support_ratio",
        ]
    ].head(30)
)


## 6. Visualize pattern reduction

These plots show how the frequent-pattern search is reduced into closed and then discriminative patterns.

The exact number depends on the support settings used in Notebook 02 and the filters above.


In [ ]:
summary = pd.DataFrame({
    "stage": [
        "Positive frequent",
        "Positive closed",
        "Negative frequent",
        "Negative closed",
        "Discriminative"
    ],
    "count": [
        len(positive_df),
        len(positive_closed),
        len(negative_df),
        len(negative_closed),
        len(discriminative_df)
    ]
})

display(summary)

plt.figure(figsize=(9, 5))
plt.bar(summary["stage"], summary["count"])
plt.ylabel("Number of patterns")
plt.title("Pattern reduction across mining stages")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "pattern_reduction.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
if len(discriminative_df) > 0:
    top_plot = discriminative_df.head(20).copy()
    top_plot = top_plot.sort_values("support_difference")

    plt.figure(figsize=(10, 7))
    plt.barh(top_plot["pattern_text"], top_plot["support_difference"])
    plt.xlabel("Positive support − negative support")
    plt.ylabel("Pattern")
    plt.title("Top discriminative sequential patterns")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "top_discriminative_patterns.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("No discriminative patterns passed the current filters.")


In [ ]:
if len(discriminative_df) > 0:
    plt.figure(figsize=(7, 5))
    plt.scatter(
        discriminative_df["negative_support"],
        discriminative_df["positive_support"],
        alpha=0.7
    )
    max_val = max(
        discriminative_df["positive_support"].max(),
        discriminative_df["negative_support"].max()
    )
    plt.plot([0, max_val], [0, max_val], linestyle="--")
    plt.xlabel("Negative support")
    plt.ylabel("Positive support")
    plt.title("Positive vs negative support of discriminative patterns")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "positive_vs_negative_support.png", dpi=200, bbox_inches="tight")
    plt.show()


## 7. Save outputs for Notebook 04

Notebook 04 will use the final discriminative pattern set to create distance-based features for each recent patient sequence.

The `pattern_key` column is retained as a Python-readable representation, while `pattern_text` provides a human-readable version for reporting.


In [ ]:
# Save closed-pattern comparisons
closed_output = PATTERN_DIR / "closed_pattern_support_comparison.csv"
closed_support_df.to_csv(closed_output, index=False)

# Save final discriminative patterns
discriminative_output = PATTERN_DIR / "discriminative_patterns.csv"
discriminative_df.to_csv(discriminative_output, index=False)

# Save a compact run summary
run_summary = pd.DataFrame({
    "parameter": [
        "MIN_DISCRIMINATIVE_SUPPORT",
        "MIN_SUPPORT_DIFFERENCE",
        "MIN_SUPPORT_RATIO",
        "MAX_DISCRIMINATIVE_PATTERNS",
        "positive_frequent_patterns",
        "positive_closed_patterns",
        "negative_frequent_patterns",
        "negative_closed_patterns",
        "final_discriminative_patterns",
    ],
    "value": [
        MIN_DISCRIMINATIVE_SUPPORT,
        MIN_SUPPORT_DIFFERENCE,
        MIN_SUPPORT_RATIO,
        MAX_DISCRIMINATIVE_PATTERNS,
        len(positive_df),
        len(positive_closed),
        len(negative_df),
        len(negative_closed),
        len(discriminative_df),
    ]
})

run_summary.to_csv(PATTERN_DIR / "closed_discriminative_run_summary.csv", index=False)

print("Saved:")
print(" -", closed_output)
print(" -", discriminative_output)
print(" -", PATTERN_DIR / "closed_discriminative_run_summary.csv")
print(" -", FIGURE_DIR)


## 8. Sanity checks

Before moving to Notebook 04, verify that:

- the frequent-pattern files were loaded successfully;
- closed-pattern counts are not unexpectedly zero;
- discriminative patterns have positive support greater than or equal to negative support;
- the final pattern count is manageable for distance computation;
- the saved `discriminative_patterns.csv` contains the patterns expected by the next notebook.


In [ ]:
assert len(positive_df) > 0, "No positive frequent patterns were loaded."
assert len(negative_df) > 0, "No negative frequent patterns were loaded."

if len(discriminative_df) > 0:
    assert (discriminative_df["positive_support"] >= discriminative_df["negative_support"]).all()
    assert (discriminative_df["positive_support"] >= MIN_DISCRIMINATIVE_SUPPORT).all()
    assert (discriminative_df["support_difference"] >= MIN_SUPPORT_DIFFERENCE).all()

print("Sanity checks passed.")
print(f"Ready for Notebook 04 with {len(discriminative_df):,} discriminative patterns.")
